Task 3 : Cleaning Dataset 

In [1]:
import pandas as pd
import numpy as np
import json

In [3]:
csv_data = pd.read_csv(r"C:\Users\patil\Downloads\AB_NYC_2019.csv\AB_NYC_2019.csv")

In [4]:
# Load JSON
with open(r"C:\Users\patil\Downloads\CA_category_id.json") as f:
    json_data = json.load(f)

json_df = pd.json_normalize(json_data)


In [7]:
# ------------------ 2. Combine Data (Optional) ------------------

# If both datasets have similar columns
df = pd.concat([csv_data, json_df], ignore_index=True)

# ------------------ 3. Basic Info ------------------
print(df.info())
print(df.head())

# ------------------ 4. Handle Missing Values ------------------

# Fill numeric missing values with mean
df.fillna(df.mean(numeric_only=True), inplace=True)

for col in df.select_dtypes(include='object'):
    mode_val = df[col].mode()

    if not mode_val.empty:
        val = mode_val.iloc[0]

        # If value is list → convert to string
        if isinstance(val, list):
            val = str(val)

        df[col] = df[col].fillna(val)
    else:
        df[col] = df[col].fillna("Unknown")
# ------------------ 5. Remove Duplicates ------------------
df.drop_duplicates(inplace=True)

# ------------------ 6. Standardize Data ------------------

# Column names clean
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Example: convert date column
# df['date'] = pd.to_datetime(df['date'])

# ------------------ 7. Outlier Removal (IQR) ------------------

Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | 
          (df > (Q3 + 1.5 * IQR))).any(axis=1)]

# ------------------ 8. Save Clean Data ------------------
df.to_csv("cleaned_data.csv", index=False)

print("✅ Data Cleaning Done Successfully!")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48896 entries, 0 to 48895
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  float64
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  float64
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  float64
 10  minimum_nights                  48895 non-null  float64
 11  number_of_reviews               48895 non-null  float64
 12  last_review                     

TypeError: unhashable type: 'list'